# STEP 2. Data Preprocessing

## 목적
본 노트북에서는 스타벅스 앱 행동 로그(`transcript.json`)를 중심으로  
이벤트 유형별로 상이한 `value` 컬럼 구조를 정규화하여,  
이후 **퍼널 분석, 코호트 분석, 이탈 예측 모델링**에 활용 가능한  
분석용 테이블을 생성하는 것을 목표로 한다.

---

## 처리 범위

### ✅ 포함되는 작업 (본 단계에서 수행)

#### 1. 구조 정규화
- 이벤트 로그의 `value` 컬럼을 해체
- `offer_id`, `reward`, `amount` 컬럼으로 분리하여 정규화

#### 2. 타입 및 논리 정합성 검증
- 이벤트 유형별로 필요한 컬럼 값이 정상적으로 존재하는지 확인
- `offer_id`와 `portfolio` 테이블 간 참조 무결성 검증

#### 3. 데이터 기본 품질 점검
- 결제 금액(`amount`) 및 보상(`reward`) 값의 음수 여부 확인
- 비현실적인 값 또는 명백한 오류 여부 점검
- 시간 컬럼(`time`, `event_time`)의 이상 여부 확인

#### 4. 결측값 해석
- 이벤트 구조에 따라 발생하는 **의도된 결측**과  
  데이터 품질 문제로 인한 **문제 결측**을 구분
- 결측이 발생하는 이유를 설명 가능한 상태로 정리

---

### ❌ 포함되지 않는 작업 (이후 단계에서 수행)
- 결측값 제거 또는 대체
- 이상치(outlier) 제거
- 고객 단위 집계 및 파생 지표 생성
- 시각화 중심의 탐색적 분석(EDA)

본 단계에서는 데이터의 **형식적 안정성과 논리적 일관성 확보**에 초점을 두며,  
값의 해석과 처리 전략은 이후 분석 단계에서 결정한다.

---


In [26]:
#환경 설정

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns   
import json
import datetime as dt



In [27]:
#데이터 로드 
profile = pd.read_json('../data/raw/profile.json', orient='records', lines=True)
portfolio = pd.read_json('../data/raw/portfolio.json', orient='records', lines=True)
transcript = pd.read_json('../data/raw/transcript.json', orient='records', lines=True)



In [28]:
#전처리를 위한 복사본 생성
df = transcript.copy()

### 01.value 컬럼 구조 정리
 value 컬럼은 이벤트 유형에 따라 서로 다른 정보를 담고 있다.
- 오퍼 이벤트: offer id
- 오퍼 완료 이벤트: offer_id, reward
- 거래 이벤트: amount

분석을 위해서는 이를 명시적인 컬럼으로 분리할 필요가 있다.

### **참고**
📌 JSON 기반 로그 데이터에서는
컬럼 내 값의 타입이 행마다 다를 수 있다.  

따라서 value 컬럼을 처리할 때는
isinstance(x, dict)를 통해 타입을 확인한 후
안전하게 key를 추출하는 방식으로 전처리를 수행한다.

In [29]:
# json 구조를 파악하여 그 안에 들어있는 offer id / offer_id 키를 모두 처리하여 하나의 offer_id 컬럼으로 통합
def extract_offer_id(x):
    if isinstance(x, dict):
        return x.get('offer id') or x.get('offer_id')
    return None
#x가 딕셔너리인지 먼저 확인하고
#맞으면 key를 꺼내고
#아니면 그냥 None을 반환한다

df['offer_id'] = df['value'].apply(extract_offer_id)

In [30]:
# reward 컬럼 생성
df['reward'] = df['value'].apply(
    lambda x: x.get('reward') if isinstance(x, dict) else None
)
#value의 각 x에 대해서 
#x가 딕셔너리이면 reward 키의 값을 반환하고
#아니면 None을 반환한다

In [31]:
# amount 컬럼 생성
df['amount'] = df['value'].apply(
    lambda x: x.get('amount') if isinstance(x, dict) else None
)

In [32]:
#이벤트별 컬럼이 정상적으로 생겼는지 확인
df.groupby('event')[['offer_id', 'reward', 'amount']].count()

,offer_id,reward,amount
event,,,
offer completed,33579,33579,0
offer received,76277,0,0
offer viewed,57725,0,0
transaction,0,0,138953


In [33]:
#결측 비율 확인 (일단 여기서는 의도된 결측임)
df[['offer_id', 'reward', 'amount']].isnull().mean()

offer_id    0.453304
reward      0.890456
amount      0.546696
dtype: float64

In [34]:
df[['offer_id', 'reward', 'amount']].isnull().sum()

offer_id    138953
reward      272955
amount      167581
dtype: int64

### 구조적 정합성 검증 결과

- 이벤트 유형별로 필요한 컬럼만 값이 존재함을 확인하였다.
- offer 이벤트에서는 offer_id 및 reward 정보가,
  transaction 이벤트에서는 amount 정보가 정상적으로 분리되었다.
- 높은 결측 비율은 이벤트 구조에 따른 의도된 결측으로,
  데이터 품질 문제는 아님을 확인하였다.

### 02.데이터 기본 품질 점검

In [35]:
# amount 값의 기본 품질 점검 (음수 있나? 극단치 있나?)
df['amount'].describe()

count    138953.000000
mean         12.777356
std          30.250529
min           0.050000
25%           2.780000
50%           8.890000
75%          18.070000
max        1062.280000
Name: amount, dtype: float64

transaction(거래)이벤트 발생 시 나타나는 amount(결제금액)를 살펴봄  
일단 우측 긴꼬리 (right-skewed) 분포, max 1062 극단치 존재 확인, 음수는 확인되지 않았음. 

In [36]:
# reward 분포 확인
display(df['reward'].describe())
print(df['reward'].value_counts(dropna=True))


count    33579.000000
mean         4.904137
std          2.886647
min          2.000000
25%          2.000000
50%          5.000000
75%          5.000000
max         10.000000
Name: reward, dtype: float64

reward
5.0     12070
2.0      9334
10.0     7019
3.0      5156
Name: count, dtype: int64


In [37]:
#offer_id 품질 점검
# transcript에 존재하는 offer_id 중 portfolio에 없는 값 확인
invalid_offer_ids = set(df['offer_id'].dropna()) - set(portfolio['id'])
len(invalid_offer_ids)

0

In [38]:
#event 컬럼 논리 일관성 체크
# transaction 이벤트에 offer_id가 있는 경우 (있으면 이상)
df[(df['event'] == 'transaction') & (df['offer_id'].notnull())].shape



(0, 7)

In [39]:
# offer 이벤트에 amount가 있는 경우 (있으면 이상)
df[(df['event'] != 'transaction') & (df['amount'].notnull())].shape

(0, 7)

행이 모두 0임을 확인했으므로, transaction 이벤트에 offer_id가 존재하거나,
offer 이벤트에 amount가 존재하는 구조적 오류는 발견되지 않았다.

In [40]:
df.head()

,person,event,value,time,offer_id,reward,amount
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN
4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN


transcript 행동 로그 테이블이므로 단일 컬럼 기준 유니크 키가 존재하지 않는 게 정상이므로 중복 체크 및 처리는 따로 하지 않음


In [ ]:
#전처리 데이터 저장할 폴더 만들기
import os
os.makedirs('../data/processed/', exist_ok=True)

In [45]:
# 전처리 데이터 저장
df.to_csv('../data/processed/transcript_preprocessed.csv', index=False)